# Milestone 4 : fiabiliser la solution et analyser les gains

## 1. Suites de tests

Les tests portent sur le moteur du milestone 3, qui est dans le package `backend/` (un module par étape) et que `main.py` enchaîne. Le notebook appelle `main.executer()` : le moteur tourne une fois (planning final, sorties 1 et 2), sans rien afficher ni écrire. Les tests appellent directement les fonctions du backend et les résultats, regroupés sous le nom `m` (`m.consolider_couple`, `m.planning_complet`, `m.sortie2`, ...).

Les tests sont écrits avec **pytest** et exécutés dans le notebook par **ipytest** : chaque cellule `%%ipytest` contient un test (une fonction `test_...` avec des `assert`), et affiche son résultat (`passed` ou `failed` avec le détail). Quand un test a plusieurs cas (`@pytest.mark.parametrize`), chaque cas compte comme un test.

Les mêmes tests existent dans le dossier `tests/` à la racine du projet, pour les lancer hors du notebook (et plus tard dans une CI) : `uv run pytest`. La partie 1.6 les relance tous et donne le bilan.

In [ ]:
import sys

import ipytest

ipytest.autoconfig()
sys.path.insert(0, "..")

import contextlib
import io
import types

import numpy as np
import pandas as pd
import pytest

with contextlib.redirect_stdout(io.StringIO()):   # le moteur tourne une fois, sans rien afficher ni écrire
    import main
    from backend import charge, config, contraintes, couts, donnees, formatage, jours_fixes, lissage, report
    R = main.executer()

# m regroupe les fonctions et données du backend (un seul nom pour les tests) et les résultats du moteur
m = types.SimpleNamespace()
for module in (config, couts, donnees, charge, contraintes, jours_fixes, lissage, report, formatage):
    vars(m).update({nom: valeur for nom, valeur in vars(module).items() if not nom.startswith("_")})
for nom in ("planning_complet", "lignes_complet", "sortie1", "sortie2"):
    setattr(m, nom, getattr(R, nom))

J = pd.Timestamp
P, L = m.planning_complet, m.lignes_complet       # planning final (une ligne par camion) et lignes de besoin livrées


def lignes_test(*lignes):
    # Lignes de besoin d'un couple fictif : (date du besoin, poids en kg, volume en m³)
    return pd.DataFrame([{"need_date": J(d), "weight": w, "volume": v} for d, w, v in lignes])

print(f"Moteur chargé : {len(P)} camions ({int(P['reserve'].sum())} réservés), {len(L)} lignes de besoin, "
      f"du {m.DEBUT:%d/%m} au {m.FIN:%d/%m}")

### 1.1 Tests unitaires des fonctions

Les fonctions clés du moteur sont testées sur des cas construits à la main, dont on connaît le résultat : grille tarifaire, date de départ des camions, consolidation d'un couple, contrôle du lissage. Les dates sont celles du planning (semaine du lundi 16/11/2026, sauf mention contraire).

#### Test 1 : `tranche` : choix de la tranche de la grille

La grille tarifaire est par tranches : on prend la première borne supérieure ou égale à la valeur (on majore). On teste une borne exacte, une valeur juste au-dessus et une valeur au-delà de la dernière borne.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("valeur, tranches, attendu", [
    (30, m.TRANCHES_KM, "30"),        # borne exacte
    (30.1, m.TRANCHES_KM, "50"),      # juste au-dessus : tranche suivante
    (650, m.TRANCHES_KM, "650"),      # dernière borne
    (651, m.TRANCHES_KM, ">650"),     # au-delà de la grille
    (0, m.TRANCHES_KG, "50"),         # poids nul : première tranche
    (4000, m.TRANCHES_KG, "4000"),
    (4000.5, m.TRANCHES_KG, ">4000"),
])
def test_tranche(valeur, tranches, attendu):
    assert m.tranche(valeur, tranches) == attendu

#### Test 2 : `cout_transport` : exemple de la rate sheet

Le prix d'un camion = coût de la grille × (1 + marge de 239,23 %) × surcharge diesel de 1,1943. La rate sheet donne un exemple chiffré : 1 000 kg à 300 km coûtent 335,34 €.

In [ ]:
%%ipytest -vv --no-header

def test_cout_transport_exemple_rate_sheet():
    assert m.cout_transport(1000, 300) == 335.34

#### Test 3 : `cout_transport` : le prix ne baisse jamais quand le poids ou la distance augmente

On calcule le prix sur toute la grille (chaque tranche de poids × chaque tranche de distance, plus les tranches « au-delà »). Un camion plus lourd ou qui va plus loin ne doit jamais coûter moins cher : sinon l'optimisation pourrait gagner de l'argent en se trompant.

In [ ]:
%%ipytest -vv --no-header

def test_cout_transport_croissant():
    poids = m.TRANCHES_KG + [m.POIDS_PAR_DEFAUT]
    distances = m.TRANCHES_KM + [m.DISTANCE_PAR_DEFAUT]
    grille = np.array([[m.cout_transport(w, d) for w in poids] for d in distances])
    assert (np.diff(grille, axis=1) >= 0).all(), "le prix baisse quand le poids augmente"
    assert (np.diff(grille, axis=0) >= 0).all(), "le prix baisse quand la distance augmente"

#### Test 4 : `cout_camion` : valeurs par défaut des données manquantes

Hypothèse des milestones 1 et 2 : un poids inconnu est chiffré dans la tranche > 4 000 kg, une distance inconnue dans la tranche > 650 km. On vérifie que `cout_camion` applique bien ces valeurs par défaut.

In [ ]:
%%ipytest -vv --no-header

def test_cout_camion_valeurs_par_defaut():
    avec_distance = next(s for s in m.distances.index if s not in m.SANS_DISTANCE)
    sans_distance = m.SANS_DISTANCE[0]
    assert m.tranche(m.POIDS_PAR_DEFAUT, m.TRANCHES_KG) == ">4000"
    assert m.tranche(m.DISTANCE_PAR_DEFAUT, m.TRANCHES_KM) == ">650"
    assert m.cout_camion(np.nan, avec_distance) == m.cout_transport(m.POIDS_PAR_DEFAUT, m.distances[avec_distance])
    assert m.cout_camion(1000, sans_distance) == m.cout_transport(1000, m.DISTANCE_PAR_DEFAUT)

#### Test 5 : `date_camion` : jour de départ du camion

Un besoin part le dernier jour fixe du couple avant (ou égal à) sa date. Trois cas : un jour fixe dans la même semaine, un profil à 2 jours, et un besoin du premier jour du planning (lundi 09/11) dont le jour fixe est le jeudi précédent : le camion part dans la semaine ajoutée avant le planning, le jeudi 05/11.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("besoin, profil, attendu", [
    ("2026-11-18", (0,), "2026-11-16"),     # besoin mercredi, livraison le lundi -> lundi 16/11
    ("2026-11-20", (0, 3), "2026-11-19"),   # besoin vendredi, livraison lundi + jeudi -> jeudi 19/11
    ("2026-11-09", (3,), "2026-11-05"),     # besoin du 1er jour, jeudi précédent : semaine ajoutée avant le planning
])
def test_date_camion(besoin, profil, attendu):
    assert m.date_camion(J(besoin), profil) == J(attendu)

#### Test 6 : `date_camion` : règle vérifiée pour tous les profils et tous les jours du planning

Au lieu de quelques exemples, on vérifie la règle pour les 25 profils (1 à 3 jours fixes) et chaque jour de l'horizon, week-end compris : le camion ne part jamais après le besoin ni plus de 7 jours avant, toujours un jour du profil (donc un jour ouvré), et jamais avant le premier jour de livraison possible (lundi 02/11).

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("profil", m.PROFILS, ids=m.nom_profil)
def test_date_camion_regle_generale(profil):
    for besoin in pd.date_range(m.DEBUT, m.FIN):
        date = m.date_camion(besoin, profil)
        assert date <= besoin, f"{besoin:%d/%m} : livré en retard"
        assert (besoin - date).days <= 7, f"{besoin:%d/%m} : plus de 7 jours d'avance"
        assert date.day_of_week in profil, f"{besoin:%d/%m} : départ hors des jours fixes"
        assert date >= m.DEBUT_LIVRAISONS, f"{besoin:%d/%m} : départ avant le premier jour de livraison"

#### Test 7 : `prochain_jour_fixe` : jour fixe suivant

Utilisée par le report (bloc 4) : premier jour du profil strictement après une date, éventuellement la semaine suivante.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("date, profil, attendu", [
    ("2026-11-16", (0, 3), "2026-11-19"),   # lundi -> jeudi de la même semaine
    ("2026-11-19", (0, 3), "2026-11-23"),   # jeudi -> lundi suivant
    ("2026-11-16", (0,), "2026-11-23"),     # un seul jour fixe : la semaine suivante
])
def test_prochain_jour_fixe(date, profil, attendu):
    assert m.prochain_jour_fixe(J(date), profil) == J(attendu)

#### Test 8 : `nom_profil` : libellé des jours fixes

Le libellé est celui de la sortie 1 (colonne des jours) : les jours dans l'ordre, séparés par « + ».

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("profil, attendu", [((2,), "Mercredi"), ((0, 3), "Lundi + Jeudi")])
def test_nom_profil(profil, attendu):
    assert m.nom_profil(profil) == attendu

#### Test 9 : `profil_utilise` : un couple à plusieurs jours fixes les utilise vraiment (C11)

Un couple au profil « mardi + vendredi » doit avoir au moins une semaine avec un camion le mardi et un camion le vendredi, sinon il n'utilise jamais ses 2 jours. On teste une semaine complète, un jour jamais utilisé, deux jours utilisés mais jamais la même semaine, et un profil d'un seul jour (toujours utilisé).

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("dates, profil, attendu", [
    (["2026-11-17", "2026-11-20"], (1, 4), True),                  # mardi + vendredi la même semaine
    (["2026-11-17", "2026-11-24"], (1, 4), False),                 # le vendredi n'est jamais utilisé
    (["2026-11-17", "2026-11-27"], (1, 4), False),                 # mardi et vendredi, mais jamais la même semaine
    (["2026-11-17", "2026-11-18", "2026-11-19"], (1, 2, 3), True), # 3 jours la même semaine
    (["2026-11-17", "2026-11-18"], (1, 2, 3), False),              # 3e jour jamais utilisé
    (["2026-11-18"], (2,), True),                                  # un seul jour : toujours utilisé
])
def test_profil_utilise(dates, profil, attendu):
    assert m.profil_utilise([J(d) for d in dates], profil) is attendu

#### Test 10 : `consolider_couple` : les besoins d'une semaine dans un seul camion

Trois petits besoins du lundi au vendredi (1 t au total), profil « lundi » : ils doivent partir ensemble, dans un seul camion, le lundi.

In [ ]:
%%ipytest -vv --no-header

def test_consolider_couple_regroupe_la_semaine():
    camions = m.consolider_couple(lignes_test(("2026-11-16", 500, 2), ("2026-11-17", 300, 1), ("2026-11-20", 200, 1)), (0,))
    assert len(camions) == 1
    assert camions["date"].iloc[0] == J("2026-11-16")
    assert camions["poids"].iloc[0] == 1000

#### Test 11 : `consolider_couple` : capacité en poids et en volume

Quand la charge dépasse 24,5 t ou 110,4 m³, un deuxième camion est ajouté (SF-07) : 3 besoins de 10 t (30 t) ou 2 besoins de 60 m³ (120 m³) demandent 2 camions, chacun sous la limite.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("cas, lignes", [
    ("poids : 3 x 10 t", [("2026-11-16", 10000, 1), ("2026-11-17", 10000, 1), ("2026-11-18", 10000, 1)]),
    ("volume : 2 x 60 m3", [("2026-11-16", 100, 60), ("2026-11-17", 100, 60)]),
])
def test_consolider_couple_capacite(cas, lignes):
    camions = m.consolider_couple(lignes_test(*lignes), (0,))
    assert len(camions) == 2
    assert (camions["poids"] <= m.POIDS_MAX).all() and (camions["volume"] <= m.VOLUME_MAX).all()

#### Test 12 : `consolider_couple` : fenêtre de 7 jours d'anticipation

Un besoin 7 jours après le camion du lundi part encore dans ce camion (limite autorisée, SF-06). À 8 jours, il part dans un camion suivant.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("jours_apres, nb_camions", [(7, 1), (8, 2)])
def test_consolider_couple_fenetre_7_jours(jours_apres, nb_camions):
    second = J("2026-11-16") + pd.Timedelta(days=jours_apres)
    camions = m.consolider_couple(lignes_test(("2026-11-16", 100, 1), (second, 100, 1)), (0,))
    assert len(camions) == nb_camions

#### Test 13 : `consolider_couple` : aucune ligne perdue ni dupliquée

40 besoins tirés au hasard (graine fixe) sur 4 semaines, profil « mardi + jeudi ». Chaque ligne doit être dans exactement un camion, les poids et volumes conservés, chaque camion sous les limites, et chaque ligne livrée entre 0 et 7 jours avant son besoin.

In [ ]:
%%ipytest -vv --no-header

def test_consolider_couple_conserve_les_lignes():
    rng = np.random.default_rng(0)
    jours_ouvres = pd.date_range("2026-11-09", "2026-12-04", freq="B")
    lignes = lignes_test(*zip(rng.choice(jours_ouvres, 40), rng.uniform(100, 9000, 40), rng.uniform(1, 40, 40)))
    lignes = lignes.sort_values(["need_date", "weight"], ascending=[True, False])
    camions = m.consolider_couple(lignes, (1, 3))
    contenu = [l for c in camions["lignes"] for l in c]
    assert sorted(contenu) == sorted(lignes.index), "ligne perdue ou dupliquée"
    assert np.isclose(camions["poids"].sum(), lignes["weight"].sum())
    assert np.isclose(camions["volume"].sum(), lignes["volume"].sum())
    assert (camions["poids"] <= m.POIDS_MAX).all() and (camions["volume"] <= m.VOLUME_MAX).all()
    for c in camions.itertuples():
        besoins = lignes.loc[c.lignes, "need_date"]
        assert (besoins >= c.date).all() and (besoins <= c.date + m.FENETRE).all()

#### Test 14 : `ecart` : écart d'un jour à la moyenne de la semaine

Mesure utilisée pour le lissage : écart du jour le plus éloigné de la moyenne, en part de la moyenne. Semaine régulière → 0 ; 3-2-2-2-1 (moyenne 2, écart 1) → 50 % ; tout le lundi → 400 % ; semaine vide → 0.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("semaine, attendu", [
    ([2, 2, 2, 2, 2], 0.0),
    ([3, 2, 2, 2, 1], 0.5),
    ([10, 0, 0, 0, 0], 4.0),
    ([0, 0, 0, 0, 0], 0.0),
])
def test_ecart(semaine, attendu):
    assert m.ecart(semaine) == pytest.approx(attendu)

#### Test 15 : `hors_tolerance` : contrôle du lissage d'une semaine

Une semaine est hors tolérance si un jour s'écarte de la moyenne de plus de 35 % et d'au moins 1 camion. Les semaines de moins de 5 camions ne sont pas contrôlées (une usine ne peut pas faire mieux).

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("semaine, attendu", [
    ([2, 2, 2, 2, 2], False),    # parfaitement régulière
    ([3, 2, 2, 2, 1], False),    # écart d'un seul camion : toléré
    ([10, 0, 0, 0, 0], True),    # tout le même jour
    ([6, 1, 1, 1, 1], True),     # pic le lundi
    ([4, 0, 0, 0, 0], False),    # moins de 5 camions : pas contrôlée
])
def test_hors_tolerance(semaine, attendu):
    assert m.hors_tolerance(np.array([semaine], dtype=float)).tolist() == [attendu]

#### Test 16 : `dispersion` : critère minimisé par le lissage

Somme des carrés des écarts à la moyenne de chaque semaine contrôlée : nulle quand chaque semaine est régulière, positive dès qu'un jour est chargé. La semaine ajoutée avant le planning et la dernière semaine (incomplètes) ne sont pas contrôlées, donc ne comptent pas.

In [ ]:
%%ipytest -vv --no-header

def test_dispersion():
    reguliere = np.full((m.NB_SEMAINES, 5), 2.0)
    assert m.dispersion(reguliere) == 0
    pic = reguliere.copy()
    pic[m.CONTROLE[0]] = [10, 0, 0, 0, 0]
    assert m.dispersion(pic) > 0
    for semaine in (0, m.NB_SEMAINES - 1):                  # semaine ajoutée et dernière semaine : non contrôlées
        non_controlee = reguliere.copy()
        non_controlee[semaine] = [10, 0, 0, 0, 0]
        assert m.dispersion(non_controlee) == 0

### 1.2 Tests du respect des contraintes

Les contraintes ont été définies au milestone 3 (partie 1). Chacune a sa fonction de vérification, qui retourne les cas qui la violent (table vide = contrainte respectée) :

| # | Contrainte | Type | Fonction |
|---|---|---|---|
| C1 | Un camion ne dépasse pas 24,5 t | dure | `c1_poids` |
| C2 | Un camion ne dépasse pas 110,4 m³ | dure | `c2_volume` |
| C3 | Chaque ligne est livrée, une seule fois, entière | dure | `c3_exhaustivite` |
| C4 | Un camion ne transporte qu'un seul couple | dure | `c4_un_couple` |
| C5 | Jamais de retard | dure | `c5_retard` |
| C6 | Au plus 7 jours d'avance | dure | `c6_avance` |
| C7 | Livraison un jour ouvré, dans l'horizon du planning | dure | `c7_jours_ouvres` |
| C8 | Jours fixes : 1 à 3 jours par couple, le camion part un de ces jours | dure | `c8_jours_fixes` |
| C9 | Au plus 1 camion par couple et par jour, sauf cas impossible (besoin d'un jour > 24,5 t, ou camion réservé d'une ligne sans poids) | dure | `c9_un_camion_par_jour` |
| C10 | Lissage : aucun jour ne s'écarte de plus de 35 % de la moyenne de la semaine | souple | `c10_lissage` |
| C11 | Profil utilisé : un couple à 2 ou 3 jours fixes livre au moins une semaine sur tous ses jours fixes | dure | `c11_profil_utilise` |

On teste deux choses : le planning final respecte les contraintes, et les fonctions de vérification détectent vraiment une erreur. Une fonction qui renverrait toujours une table vide passerait le premier test sans rien vérifier : pour chaque contrainte, on introduit donc volontairement une erreur dans une copie du planning, et on vérifie qu'elle est signalée.

La cellule ci-dessous prépare ces erreurs (une par contrainte).

In [ ]:
FONCTIONS = {code_c: fonction for code_c, _, _, fonction in m.CONTRAINTES}
DURES = [code_c for code_c, _, type_c, _ in m.CONTRAINTES if type_c == "dure"]


def violations(code_c, planning, lignes):
    # Violations d'une contrainte (pour C9 : seulement les cas évitables, sans cause justifiée)
    args = (planning, lignes, m.IDS_ATTENDUS) if code_c == "C3" else (planning, lignes)
    v = FONCTIONS[code_c](*args)
    return v[~v["justifie"]] if code_c == "C9" else v


# ---------- Erreurs introduites volontairement, une par contrainte ----------
opti = P[~P["reserve"]]
i0 = opti.index[0]                                                          # un camion optimisé quelconque
multi = L.groupby("camion_id").size().loc[lambda n: n > 1].index[0]         # un camion qui transporte plusieurs lignes
couples_c9 = set(map(tuple, m.c9_un_camion_par_jour(P, L)[["plant", "supplier_id"]].values))
i9 = next(i for i, r in opti.iterrows() if (r["plant"], r["supplier_id"]) not in couples_c9)
i8 = opti.index[0]
un_jour = opti[opti["profil"].map(len) == 1].iloc[0]                       # un couple à 1 jour fixe (pour C11)


def avec_erreur(code_c):
    # Copie du planning et des lignes avec une erreur qui viole la contrainte
    p, l = P.copy(), L.copy()
    if code_c == "C1":
        p.loc[i0, "poids"] = m.POIDS_MAX + 1                                    # camion de 24,5 t + 1 kg
    elif code_c == "C2":
        p.loc[i0, "volume"] = m.VOLUME_MAX + 1                                  # camion de 110,4 m³ + 1 m³
    elif code_c == "C3":
        l = l.drop(index=l.index[0])                                            # ligne oubliée
    elif code_c == "C4":
        l.loc[l.index[l["camion_id"] == multi][0], "supplier_id"] = "SUP_99999"  # ligne d'un autre fournisseur
    elif code_c == "C5":
        l.loc[l.index[0], "date_livraison"] = l.loc[l.index[0], "need_date"] + pd.Timedelta(days=1)   # 1 jour de retard
    elif code_c == "C6":
        l.loc[l.index[0], "date_livraison"] = l.loc[l.index[0], "need_date"] - pd.Timedelta(days=8)   # 8 jours d'avance
    elif code_c == "C7":
        p.loc[i0, "date"] = J("2026-11-21")                                     # camion un samedi
    elif code_c == "C8":
        date, profil = p.loc[i8, "date"], p.loc[i8, "profil"]                   # camion hors de ses jours fixes
        p.loc[i8, "date"] = date + pd.Timedelta(days=next(j for j in range(5) if j not in profil) - date.day_of_week)
    elif code_c == "C9":
        p = pd.concat([p, p.loc[[i9]]], ignore_index=True)                      # 2 camions le même jour sans raison
    elif code_c == "C11":                                                       # profil de 2 jours dont un jamais utilisé
        autre = next(j for j in range(5) if j not in un_jour["profil"])
        meme_couple = (p["plant"] == un_jour["plant"]) & (p["supplier_id"] == un_jour["supplier_id"]) & ~p["reserve"]
        p["profil"] = [tuple(sorted({*prof, autre})) if cible else prof for prof, cible in zip(p["profil"], meme_couple)]
    elif code_c == "C10":
        renfort = p[p["plant"] == m.USINES[0]].head(1).assign(date=J("2026-11-16"))
        p = pd.concat([p] + [renfort] * 15, ignore_index=True)                  # 15 camions de plus un lundi
    return p, l

#### Test 17 : Le planning final respecte chaque contrainte dure (C1 à C9 et C11)

On rejoue les fonctions de vérification du moteur sur le planning produit par `main.executer()` : aucune violation, pour chacune des 10 contraintes dures (un cas de test par contrainte). Pour C9, les 2 camions d'un même jour dus à un cas impossible (besoin de plus de 24,5 t, ou camion réservé d'une ligne sans poids) ne sont pas des violations.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("contrainte", DURES)
def test_contrainte_dure_respectee(contrainte):
    v = violations(contrainte, P, L)
    assert v.empty, f"{contrainte} : {len(v)} violation(s)\n{v.head()}"

#### Test 18 : C10 lissage : au plus 2 semaines hors tolérance

Le lissage est une contrainte souple : avec le moteur actuel, 2 semaines sur 20 restent hors tolérance. Le test empêche une régression (plus de semaines hors tolérance qu'aujourd'hui).

In [ ]:
%%ipytest -vv --no-header

def test_lissage_c10():
    hors = violations("C10", P, L)
    assert len(hors) <= 2, f"{len(hors)} semaines hors tolérance\n{hors}"

#### Test 19 : C11 : chaque couple à plusieurs jours fixes livre au moins une semaine sur tous ses jours

Contrôle indépendant de la fonction `c11_profil_utilise` : on recalcule, couple par couple, les jours livrés chaque semaine à partir du planning, et on cherche une semaine qui contient tous les jours fixes du profil. Un profil à 2 ou 3 jours qu'on n'utilise jamais en entier serait un faux profil (en réalité, un profil moins cher à moins de jours).

In [ ]:
%%ipytest -vv --no-header

def test_c11_profil_utilise_une_semaine_complete():
    optimises = P[~P["reserve"]]
    assert (optimises["profil"].map(len) >= 2).any(), "aucun couple à plusieurs jours fixes : test sans objet"
    for (plant, supplier), camions in optimises.groupby(["plant", "supplier_id"]):
        profil = set(camions["profil"].iloc[0])
        jours_par_semaine = camions.groupby(camions["date"].dt.isocalendar().week)["date"].apply(
            lambda d: set(d.dt.day_of_week))
        assert any(profil <= jours for jours in jours_par_semaine), (plant, supplier, sorted(profil))

#### Test 20 : Chaque fonction de vérification détecte l'erreur qu'on lui introduit

Pour chaque contrainte, l'erreur préparée plus haut (camion de 24,5 t + 1 kg, ligne en retard, camion un samedi, profil à 2 jours dont un jamais utilisé, ...) doit faire apparaître au moins une violation de plus que sur le planning correct.

In [ ]:
%%ipytest -vv --no-header

@pytest.mark.parametrize("contrainte", list(FONCTIONS))
def test_verification_detecte_erreur(contrainte):
    assert len(violations(contrainte, *avec_erreur(contrainte))) > len(violations(contrainte, P, L))

#### Test 21 : Cohérence du planning : contenu, poids et coût des camions

`valider_resultats` (milestone 3) vérifie que chaque camion contient ce qu'il annonce (poids, volume), que le poids total livré est celui des besoins, qu'il y a un camion réservé par ligne sans poids, et que le coût de chaque camion suit la grille tarifaire.

In [ ]:
%%ipytest -vv --no-header

def test_planning_coherent():
    controles = m.valider_resultats(P, L)
    echecs = controles[controles["Résultat"] != "OK"].index.tolist()
    assert not echecs, echecs

#### Test 22 : C9 : 2 camions le même jour seulement dans un cas impossible

Un couple ne reçoit 2 camions le même jour que dans deux cas : son besoin d'un seul jour dépasse 24,5 t (il ne tient pas dans un camion), ou une de ses lignes sans poids a son camion réservé ce jour-là (règle absolue : elle ne peut être ni regroupée ni déplacée). On recalcule ces cas à partir des lignes livrées.

In [ ]:
%%ipytest -vv --no-header

def test_c9_doublons_justifies():
    doublons = m.c9_un_camion_par_jour(P, L)
    besoin_jour = L.dropna(subset=["weight"]).groupby(["plant", "supplier_id", "need_date"])["weight"].sum()
    couples_lourds = set(besoin_jour[besoin_jour > m.POIDS_MAX].index.droplevel("need_date"))
    optimises = P[~P["reserve"]].groupby(["plant", "supplier_id", "date"]).size()
    for r in doublons.itertuples():
        nb_optimises = optimises.get((r.plant, r.supplier_id, r.date), 0)
        assert nb_optimises <= 1 or (r.plant, r.supplier_id) in couples_lourds, (r.plant, r.supplier_id, r.date)

### 1.3 Cas limites et hypothèses de données manquantes

Les hypothèses prises aux milestones 1 à 3 pour les données incomplètes et les cas particuliers sont vérifiées sur le planning final :
- ligne de plus de 24,5 t : exclue du moteur, en attendant la réponse du client ;
- poids manquant : la ligne ne peut pas être consolidée, elle a son propre camion réservé, le jour du besoin, chiffré au tarif > 4 000 kg ;
- distance manquante : le camion est chiffré au tarif > 650 km, et la distance est vide (« inconnue ») dans la sortie 1 ;
- début du planning : les premiers besoins partent le dernier jour fixe de leur couple, dans la semaine ajoutée avant le planning (à partir du lundi 02/11) ;
- quantité nulle (SF-02) et couple à une seule ligne.

In [ ]:
q = m.quarantaine

#### Test 23 : Ligne de plus de 24,5 t : exclue du moteur

La seule ligne de la quarantaine de plus de 24,5 t ne peut pas tenir dans un camion : elle ne doit apparaître dans aucun camion.

In [ ]:
%%ipytest -vv --no-header

def test_ligne_trop_lourde_exclue():
    lourdes = q[q["weight"] > m.POIDS_MAX]
    assert len(lourdes) == 1, f"{len(lourdes)} ligne(s) de plus de 24,5 t"
    cles = ["need_date", "supplier_id", "material", "plant", "unloading_point"]
    assert lourdes[cles].merge(L[cles]).empty, "la ligne lourde est livrée"

#### Test 24 : Poids manquant : un camion réservé par ligne

Chaque ligne sans poids a son propre camion réservé, livré le jour du besoin et chiffré dans la tranche > 4 000 kg.

In [ ]:
%%ipytest -vv --no-header

def test_poids_manquant_camion_reserve():
    reserves, lignes_r = P[P["reserve"]], L[L["reserve"]]
    assert len(reserves) == q["weight"].isna().sum() == len(lignes_r)
    assert (lignes_r.groupby("camion_id").size() == 1).all()
    assert (lignes_r["date_livraison"] == lignes_r["need_date"]).all()
    attendu = [m.cout_transport(m.POIDS_PAR_DEFAUT, m.distances[s]) for s in reserves["supplier_id"]]
    assert np.allclose(reserves["cout"], attendu)

#### Test 25 : Distance manquante : tarif > 650 km, distance vide dans la sortie 1

Les fournisseurs sans distance sont chiffrés au tarif le plus élevé, pour ne pas surestimer l'économie. Dans la sortie 1, leur distance reste vide (et non 651 km, valeur technique).

In [ ]:
%%ipytest -vv --no-header

def test_distance_manquante():
    camions = P[P["supplier_id"].isin(m.SANS_DISTANCE)]
    assert len(camions) > 0, "aucun fournisseur sans distance dans le planning"
    poids = camions["poids"].fillna(m.POIDS_PAR_DEFAUT)
    assert np.allclose(camions["cout"], [m.cout_transport(w, m.DISTANCE_PAR_DEFAUT) for w in poids])
    assert m.sortie1.loc[m.sortie1["Fournisseur"].isin(m.SANS_DISTANCE), "Distance"].isna().all()

#### Test 26 : Début du planning : semaine ajoutée avant le planning

Les premiers besoins (du lundi 09/11) ont leur jour fixe avant le début du planning : leurs camions partent dans la semaine ajoutée, à partir du lundi 02/11. On vérifie que ces camions existent, qu'aucun ne part avant le 02/11, et qu'ils ne transportent que des besoins du planning.

In [ ]:
%%ipytest -vv --no-header

def test_semaine_ajoutee_avant_le_planning():
    avant = P[P["date"] < m.DEBUT]
    assert len(avant) > 0
    assert (avant["date"] >= m.DEBUT_LIVRAISONS).all(), "camion avant le premier jour de livraison"
    assert not avant["reserve"].any()
    premiere = L[L["camion_id"].isin(avant["camion_id"])].groupby("camion_id")["need_date"].min()
    assert (premiere >= m.DEBUT).all(), "besoin hors planning dans un camion de la semaine ajoutée"

#### Test 27 : SF-02 : aucune ligne de quantité nulle

Le planning brut (bronze) contient des lignes de quantité 0 : elles ont été retirées au milestone 1 et ne doivent être ni livrées ni présentes dans la sortie 2.

In [ ]:
%%ipytest -vv --no-header

def test_quantite_nulle_exclue():
    brut = pd.read_csv(m.DATA / "bronze" / "released_schedule.csv")
    assert (brut["schedule_quantity"] <= 0).any(), "le cas n'existe pas dans les données brutes"
    assert (L["schedule_quantity"] > 0).all()
    assert (m.sortie2["Quantité"] > 0).all()

#### Test 28 : Couple à une seule ligne : un seul camion

Cas minimal de la consolidation : un couple qui n'a qu'un besoin sur toute la période reçoit exactement un camion, qui contient cette ligne.

In [ ]:
%%ipytest -vv --no-header

def test_couple_a_une_seule_ligne():
    assert len(m.consolider_couple(lignes_test(("2026-11-18", 500, 2)), (0,))) == 1
    nb_lignes = m.besoins.groupby(["plant", "supplier_id"]).size()
    seuls = nb_lignes[nb_lignes == 1].index
    assert len(seuls) > 0
    for plant, supplier in seuls:
        camions = P[(P["plant"] == plant) & (P["supplier_id"] == supplier) & ~P["reserve"]]
        assert len(camions) == 1 and camions["nb_lignes"].iloc[0] == 1, (plant, supplier)

### 1.4 Exhaustivité des matériaux livrés et anticipation des besoins

Le moteur réorganise les livraisons (il regroupe des besoins de plusieurs jours dans un même camion), mais il ne doit rien perdre, rien inventer et ne jamais mettre une usine en rupture :
- exhaustivité : chaque ligne de besoin du planning d'entrée (silver + quarantaine) se retrouve dans un camion, une seule fois ;
- matériaux : pour chaque material, la quantité demandée par l'usine est égale à la quantité chargée dans les camions (sortie 2) ;
- anticipation : chaque ligne arrive au plus tard le jour de son besoin (pas de rupture), et au plus 7 jours avant (pas plus d'une semaine de stock).

La cellule ci-dessous prépare la comparaison entre le planning d'entrée et la sortie 2.

In [ ]:
CLES = ["plant", "supplier_id", "material", "unloading_point"]
entree = pd.concat([m.besoins, m.lignes_reserves])                          # planning d'entrée : besoins à livrer
demande = entree.groupby(CLES)["schedule_quantity"].sum()                   # quantité demandée par material
livre = (m.sortie2.rename(columns={"usine": "plant", "fournisseur": "supplier_id", "Material": "material", "Quai": "unloading_point"})
         .groupby(CLES)["Quantité"].sum())                                  # quantité chargée dans les camions (sortie 2)

#### Test 29 : Aucune ligne oubliée ni livrée deux fois

Les lignes livrées sont exactement celles du planning d'entrée, chacune une fois et dans un camion. La sortie 2 compte toutes ces lignes et tous les camions.

In [ ]:
%%ipytest -vv --no-header

def test_aucune_ligne_oubliee_ni_doublee():
    assert L["id_ligne"].is_unique
    assert set(L["id_ligne"]) == m.IDS_ATTENDUS
    assert L["camion_id"].notna().all()
    assert m.sortie2["nb_lignes"].sum() == len(m.IDS_ATTENDUS)
    assert set(m.sortie2["camion_id"]) == set(P["camion_id"])

#### Test 30 : Chaque material est livré en quantité exacte

Pour chaque combinaison material × quai × couple, la quantité demandée (planning d'entrée) est égale à la quantité chargée dans les camions (sortie 2) : le regroupement change les jours, pas les quantités.

In [ ]:
%%ipytest -vv --no-header

def test_quantite_exacte_par_material():
    assert set(demande.index) == set(livre.index), "material manquant ou inventé"
    ecarts = demande.sub(livre, fill_value=0).abs()
    assert (ecarts < 1e-6).all(), f"{(ecarts >= 1e-6).sum()} materials avec un écart"

#### Test 31 : Le poids total livré est égal au poids demandé

Contrôle global sur les lignes au poids connu : la somme des poids livrés (lignes et sortie 2) est égale à celle des besoins. La sortie 2 arrondit les poids à 0,1 kg, d'où la petite tolérance.

In [ ]:
%%ipytest -vv --no-header

def test_poids_total_conserve():
    poids_demande = m.besoins["weight"].sum()
    assert np.isclose(L["weight"].sum(), poids_demande)
    assert abs(m.sortie2["Poids (kg)"].sum() - poids_demande) <= 0.05 * len(m.sortie2)

#### Test 32 : Anticipation : chaque ligne livrée entre 0 et 7 jours avant son besoin

Aucune usine en rupture (jamais de retard) et pas plus d'une semaine de stock (au plus 7 jours d'avance), pour chacune des lignes livrées.

In [ ]:
%%ipytest -vv --no-header

def test_anticipation_des_besoins():
    avance = (L["need_date"] - L["date_livraison"]).dt.days
    assert (avance == L["avance_jours"]).all()
    assert avance.between(0, 7).all(), avance.describe().to_string()

Le graphique montre avec combien de jours d'avance chaque ligne est livrée, du « jour même » à « 7 jours avant », la limite autorisée.

In [ ]:
import matplotlib.pyplot as plt

avance = L["avance_jours"].value_counts().sort_index().reindex(range(8), fill_value=0)
part = avance / avance.sum()
fig, ax = plt.subplots(figsize=(9, 4.5))
barres = ax.bar(avance.index, part * 100, color="royalblue", edgecolor="black")
ax.bar_label(barres, labels=[f"{m.pourcent(p)}\n({n} lignes)" for p, n in zip(part, avance)], fontsize=8, padding=2)
ax.set_xticks(range(8), ["le jour\nmême"] + [f"{j} j avant" for j in range(1, 8)])
ax.set_xlabel("Date de livraison par rapport à la date du besoin")
ax.set_ylabel("Part des lignes de besoin (%)")
ax.set_title(f"Avance des livraisons sur le besoin ({len(L)} lignes) : jamais de retard, jamais plus de 7 jours")
ax.margins(y=0.2)
ax.grid(axis="y", alpha=0.3)
plt.show()

#### Test 33 : Chaque camion part avant tous les besoins qu'il transporte

Vu côté camion : une seule date de livraison par camion, au plus tard le jour du premier besoin qu'il transporte, et au plus 7 jours avant le dernier.

In [ ]:
%%ipytest -vv --no-header

def test_camion_dans_la_fenetre_de_ses_besoins():
    par_camion = L.groupby("camion_id").agg(premier=("need_date", "min"), dernier=("need_date", "max"),
                                            date=("date_livraison", "first"), nb_dates=("date_livraison", "nunique"))
    assert (par_camion["nb_dates"] == 1).all(), "un camion avec plusieurs dates de livraison"
    assert (par_camion["date"] <= par_camion["premier"]).all()
    assert ((par_camion["dernier"] - par_camion["date"]).dt.days <= 7).all()
    assert (par_camion["date"] == P.set_index("camion_id").loc[par_camion.index, "date"]).all()

### 1.5 Format des livrables (`data/gold`)

Les deux livrables sont relus depuis les fichiers CSV de `data/gold`, comme le ferait l'utilisateur :
- `sortie1_profils.csv` : le profil de chaque couple (fournisseur, usine, distance, livraisons par semaine actuelles, fréquence cible, jours, camions par semaine, remplissage poids) ;
- `sortie2_camions.csv` : le contenu de chaque camion (material, quai, besoins regroupés, quantité, poids), une ligne par material d'un camion.

Les tests vérifient le contenu, pas la mise en forme : le CSV garde des valeurs brutes (121.0 et non « 121 km », 0.476 et non « 47,6 % »), la mise en forme est faite à l'affichage. Les besoins regroupés d'un material sont une liste de dates dans une seule cellule (par exemple « 2026-11-16, 2026-11-17 »).

Les noms de colonnes attendus sont définis ci-dessous : s'ils changent, il suffit de les modifier ici.

In [ ]:
SORTIE1 = m.SORTIE / "sortie1_profils.csv"
SORTIE2 = m.SORTIE / "sortie2_camions.csv"
COLONNES_SORTIE1 = ["Fournisseur", "Usine", "Distance", "Livraisons / sem. actuelles", "Fréquence cible", "Jours",
                    "Camions / sem.", "Remplissage poids"]
COLONNES_SORTIE2 = ["camion_id", "usine", "fournisseur", "date_livraison", "jour", "Material", "Quai", "Besoins regroupés",
                    "nb_lignes", "Quantité", "Poids (kg)", "reserve"]


def lire_sortie1():
    assert SORTIE1.exists(), f"{SORTIE1} absent : lancer python main.py"
    return pd.read_csv(SORTIE1)


def lire_sortie2():
    assert SORTIE2.exists(), f"{SORTIE2} absent : lancer python main.py"
    return pd.read_csv(SORTIE2, parse_dates=["date_livraison"], date_format="%Y-%m-%d")

#### Test 34 : Sortie 1 : colonnes et une ligne par couple

Le fichier existe, a les colonnes attendues dans l'ordre, et contient exactement une ligne (un profil) par couple fournisseur × usine du planning.

In [ ]:
%%ipytest -vv --no-header

def test_sortie1_structure():
    s1 = lire_sortie1()
    assert list(s1.columns) == COLONNES_SORTIE1
    assert not s1.duplicated(["Fournisseur", "Usine"]).any(), "couple en double"
    assert len(s1) == m.besoins.groupby(["plant", "supplier_id"]).ngroups

#### Test 35 : Sortie 1 : valeurs valides

Identifiants au bon format, distance positive (vide seulement pour un fournisseur sans distance), fréquence de 1 à 3, jours fixes valides, distincts, dans l'ordre de la semaine et en nombre égal à la fréquence, camions par semaine positifs, remplissage entre 0 et 100 %.

In [ ]:
%%ipytest -vv --no-header

def test_sortie1_valeurs():
    s1 = lire_sortie1()
    assert s1["Fournisseur"].str.fullmatch(r"SUP_\d{5}").all()
    assert s1["Usine"].str.fullmatch(r"PLT_\d{3}").all()
    sans = s1["Fournisseur"].isin(m.SANS_DISTANCE)
    assert s1.loc[sans, "Distance"].isna().all() and (s1.loc[~sans, "Distance"] > 0).all()
    assert (s1["Livraisons / sem. actuelles"] > 0).all()
    assert s1["Fréquence cible"].isin(range(1, m.JOURS_FIXES_MAX + 1)).all()
    jours = s1["Jours"].str.split(" + ", regex=False)
    assert jours.map(lambda j: set(j) <= set(m.JOURS)).all(), "jour inconnu"
    assert jours.map(lambda j: [m.JOURS.index(x) for x in j] == sorted({m.JOURS.index(x) for x in j})).all()
    assert (jours.map(len) == s1["Fréquence cible"]).all(), "nombre de jours différent de la fréquence"
    assert (s1["Camions / sem."] > 0).all()
    assert s1["Remplissage poids"].between(0, 1, inclusive="right").all()

#### Test 36 : Sortie 2 : colonnes, types et valeurs

Colonnes attendues, numéro de camion et nombre de lignes entiers, dates au format AAAA-MM-JJ et jour de la semaine cohérent, identifiants au bon format, quantité positive, poids renseigné sauf pour un camion réservé, et une seule ligne par material × quai dans un camion.

In [ ]:
%%ipytest -vv --no-header

def test_sortie2_structure():
    s2 = lire_sortie2()
    assert list(s2.columns) == COLONNES_SORTIE2
    assert pd.api.types.is_integer_dtype(s2["camion_id"]) and pd.api.types.is_integer_dtype(s2["nb_lignes"])
    assert pd.api.types.is_datetime64_any_dtype(s2["date_livraison"]) and s2["date_livraison"].notna().all()
    assert (s2["jour"] == s2["date_livraison"].dt.day_of_week.map(dict(enumerate(m.JOURS)))).all()
    assert s2["usine"].str.fullmatch(r"PLT_\d{3}").all() and s2["fournisseur"].str.fullmatch(r"SUP_\d{5}").all()
    assert s2["Material"].str.fullmatch(r"MAT_\d{6}").all() and s2["Quai"].str.fullmatch(r"ULP_\d{4}").all()
    assert (s2["Quantité"] > 0).all() and (s2["nb_lignes"] >= 1).all()
    assert s2["reserve"].dtype == bool
    assert s2.loc[s2["reserve"], "Poids (kg)"].isna().all()
    assert (s2.loc[~s2["reserve"], "Poids (kg)"] >= 0).all()
    assert not s2.duplicated(["camion_id", "Material", "Quai"]).any()

#### Test 37 : Sortie 2 : besoins regroupés dans une seule cellule

La colonne des besoins regroupés contient, dans une seule cellule, la liste des dates de besoin du material livrées par ce camion. Chaque liste doit être lisible (dates valides, distinctes, triées), compter au plus autant de dates que de lignes, et chaque date doit être entre le jour de livraison et 7 jours après.

In [ ]:
%%ipytest -vv --no-header

def test_sortie2_besoins_regroupes():
    s2 = lire_sortie2()
    erreurs = []
    for i, (besoins, livraison, nb) in enumerate(zip(s2["Besoins regroupés"], s2["date_livraison"], s2["nb_lignes"])):
        dates = pd.to_datetime(besoins.split(", "), format="%Y-%m-%d")
        if not (1 <= len(dates) <= nb and dates.is_unique and dates.is_monotonic_increasing):
            erreurs.append((i, besoins, "liste mal formée"))
        elif dates.min() < livraison or (dates.max() - livraison).days > 7:
            erreurs.append((i, besoins, f"hors fenêtre de la livraison du {livraison:%d/%m}"))
    assert not erreurs, erreurs[:5]

#### Test 38 : Les fichiers sont à jour avec le moteur

Les CSV relus doivent être identiques aux sorties calculées par `main.py` : sinon le livrable est périmé (moteur modifié sans relancer l'export).

In [ ]:
%%ipytest -vv --no-header

def test_livrables_a_jour():
    pd.testing.assert_frame_equal(lire_sortie1(), m.sortie1, check_dtype=False)
    pd.testing.assert_frame_equal(lire_sortie2(), m.sortie2, check_dtype=False)

### 1.6 Résultats des tests

On relance toute la suite du dossier `tests/` avec pytest (les mêmes tests que ci-dessus, un fichier par section). Le tableau résume chaque test : **OK** si tous ses cas passent, **NOK** sinon (le détail de l'échec est affiché sous le tableau).

In [ ]:
DESCRIPTIONS = [(1, '1.1', 'test_tranche', 'tranche : choix de la tranche de la grille', 'La grille tarifaire est par tranches : on prend la première borne supérieure ou égale à la valeur (on majore).'), (2, '1.1', 'test_cout_transport_exemple_rate_sheet', 'cout_transport : exemple de la rate sheet', "Le prix d'un camion = coût de la grille × (1 + marge de 239,23 %) × surcharge diesel de 1,1943."), (3, '1.1', 'test_cout_transport_croissant', 'cout_transport : le prix ne baisse jamais quand le poids ou la distance augmente', 'On calcule le prix sur toute la grille (chaque tranche de poids × chaque tranche de distance, plus les tranches « au-delà »).'), (4, '1.1', 'test_cout_camion_valeurs_par_defaut', 'cout_camion : valeurs par défaut des données manquantes', 'Hypothèse des milestones 1 et 2 : un poids inconnu est chiffré dans la tranche > 4 000 kg, une distance inconnue dans la tranche > 650 km.'), (5, '1.1', 'test_date_camion', 'date_camion : jour de départ du camion', 'Un besoin part le dernier jour fixe du couple avant (ou égal à) sa date.'), (6, '1.1', 'test_date_camion_regle_generale', 'date_camion : règle vérifiée pour tous les profils et tous les jours du planning', "Au lieu de quelques exemples, on vérifie la règle pour les 25 profils (1 à 3 jours fixes) et chaque jour de l'horizon, week-end compris : le camion ne part jamais après le besoin ni plus de 7 jours avant, toujours un jour du profil (donc un jour ouvré), et jamais avant le premier jour de livraison possible (lundi 02/11)."), (7, '1.1', 'test_prochain_jour_fixe', 'prochain_jour_fixe : jour fixe suivant', 'Utilisée par le report (bloc 4) : premier jour du profil strictement après une date, éventuellement la semaine suivante.'), (8, '1.1', 'test_nom_profil', 'nom_profil : libellé des jours fixes', "Le libellé est celui de la sortie 1 (colonne des jours) : les jours dans l'ordre, séparés par « + »."), (9, '1.1', 'test_profil_utilise', 'profil_utilise : un couple à plusieurs jours fixes les utilise vraiment (C11)', "Un couple au profil « mardi + vendredi » doit avoir au moins une semaine avec un camion le mardi et un camion le vendredi, sinon il n'utilise jamais ses 2 jours."), (10, '1.1', 'test_consolider_couple_regroupe_la_semaine', "consolider_couple : les besoins d'une semaine dans un seul camion", 'Trois petits besoins du lundi au vendredi (1 t au total), profil « lundi » : ils doivent partir ensemble, dans un seul camion, le lundi.'), (11, '1.1', 'test_consolider_couple_capacite', 'consolider_couple : capacité en poids et en volume', 'Quand la charge dépasse 24,5 t ou 110,4 m³, un deuxième camion est ajouté (SF-07) : 3 besoins de 10 t (30 t) ou 2 besoins de 60 m³ (120 m³) demandent 2 camions, chacun sous la limite.'), (12, '1.1', 'test_consolider_couple_fenetre_7_jours', "consolider_couple : fenêtre de 7 jours d'anticipation", 'Un besoin 7 jours après le camion du lundi part encore dans ce camion (limite autorisée, SF-06).'), (13, '1.1', 'test_consolider_couple_conserve_les_lignes', 'consolider_couple : aucune ligne perdue ni dupliquée', '40 besoins tirés au hasard (graine fixe) sur 4 semaines, profil « mardi + jeudi ».'), (14, '1.1', 'test_ecart', "ecart : écart d'un jour à la moyenne de la semaine", 'Mesure utilisée pour le lissage : écart du jour le plus éloigné de la moyenne, en part de la moyenne.'), (15, '1.1', 'test_hors_tolerance', "hors_tolerance : contrôle du lissage d'une semaine", "Une semaine est hors tolérance si un jour s'écarte de la moyenne de plus de 35 % et d'au moins 1 camion."), (16, '1.1', 'test_dispersion', 'dispersion : critère minimisé par le lissage', "Somme des carrés des écarts à la moyenne de chaque semaine contrôlée : nulle quand chaque semaine est régulière, positive dès qu'un jour est chargé."), (17, '1.2', 'test_contrainte_dure_respectee', 'Le planning final respecte chaque contrainte dure (C1 à C9 et C11)', 'On rejoue les fonctions de vérification du moteur sur le planning produit par main.executer() : aucune violation, pour chacune des 10 contraintes dures (un cas de test par contrainte).'), (18, '1.2', 'test_lissage_c10', 'C10 lissage : au plus 2 semaines hors tolérance', 'Le lissage est une contrainte souple : avec le moteur actuel, 2 semaines sur 20 restent hors tolérance.'), (19, '1.2', 'test_c11_profil_utilise_une_semaine_complete', 'C11 : chaque couple à plusieurs jours fixes livre au moins une semaine sur tous ses jours', 'Contrôle indépendant de la fonction c11_profil_utilise : on recalcule, couple par couple, les jours livrés chaque semaine à partir du planning, et on cherche une semaine qui contient tous les jours fixes du profil.'), (20, '1.2', 'test_verification_detecte_erreur', "Chaque fonction de vérification détecte l'erreur qu'on lui introduit", "Pour chaque contrainte, l'erreur préparée plus haut (camion de 24,5 t + 1 kg, ligne en retard, camion un samedi, profil à 2 jours dont un jamais utilisé, ...) doit faire apparaître au moins une violation de plus que sur le planning correct."), (21, '1.2', 'test_planning_coherent', 'Cohérence du planning : contenu, poids et coût des camions', "valider_resultats (milestone 3) vérifie que chaque camion contient ce qu'il annonce (poids, volume), que le poids total livré est celui des besoins, qu'il y a un camion réservé par ligne sans poids, et que le coût de chaque camion suit la grille tarifaire."), (22, '1.2', 'test_c9_doublons_justifies', 'C9 : 2 camions le même jour seulement dans un cas impossible', "Un couple ne reçoit 2 camions le même jour que dans deux cas : son besoin d'un seul jour dépasse 24,5 t (il ne tient pas dans un camion), ou une de ses lignes sans poids a son camion réservé ce jour-là (règle absolue : elle ne peut être ni regroupée ni déplacée)."), (23, '1.3', 'test_ligne_trop_lourde_exclue', 'Ligne de plus de 24,5 t : exclue du moteur', 'La seule ligne de la quarantaine de plus de 24,5 t ne peut pas tenir dans un camion : elle ne doit apparaître dans aucun camion.'), (24, '1.3', 'test_poids_manquant_camion_reserve', 'Poids manquant : un camion réservé par ligne', 'Chaque ligne sans poids a son propre camion réservé, livré le jour du besoin et chiffré dans la tranche > 4 000 kg.'), (25, '1.3', 'test_distance_manquante', 'Distance manquante : tarif > 650 km, distance vide dans la sortie 1', "Les fournisseurs sans distance sont chiffrés au tarif le plus élevé, pour ne pas surestimer l'économie."), (26, '1.3', 'test_semaine_ajoutee_avant_le_planning', 'Début du planning : semaine ajoutée avant le planning', 'Les premiers besoins (du lundi 09/11) ont leur jour fixe avant le début du planning : leurs camions partent dans la semaine ajoutée, à partir du lundi 02/11.'), (27, '1.3', 'test_quantite_nulle_exclue', 'SF-02 : aucune ligne de quantité nulle', 'Le planning brut (bronze) contient des lignes de quantité 0 : elles ont été retirées au milestone 1 et ne doivent être ni livrées ni présentes dans la sortie 2.'), (28, '1.3', 'test_couple_a_une_seule_ligne', 'Couple à une seule ligne : un seul camion', "Cas minimal de la consolidation : un couple qui n'a qu'un besoin sur toute la période reçoit exactement un camion, qui contient cette ligne."), (29, '1.4', 'test_aucune_ligne_oubliee_ni_doublee', 'Aucune ligne oubliée ni livrée deux fois', "Les lignes livrées sont exactement celles du planning d'entrée, chacune une fois et dans un camion."), (30, '1.4', 'test_quantite_exacte_par_material', 'Chaque material est livré en quantité exacte', "Pour chaque combinaison material × quai × couple, la quantité demandée (planning d'entrée) est égale à la quantité chargée dans les camions (sortie 2) : le regroupement change les jours, pas les quantités."), (31, '1.4', 'test_poids_total_conserve', 'Le poids total livré est égal au poids demandé', 'Contrôle global sur les lignes au poids connu : la somme des poids livrés (lignes et sortie 2) est égale à celle des besoins.'), (32, '1.4', 'test_anticipation_des_besoins', 'Anticipation : chaque ligne livrée entre 0 et 7 jours avant son besoin', "Aucune usine en rupture (jamais de retard) et pas plus d'une semaine de stock (au plus 7 jours d'avance), pour chacune des lignes livrées."), (33, '1.4', 'test_camion_dans_la_fenetre_de_ses_besoins', "Chaque camion part avant tous les besoins qu'il transporte", "Vu côté camion : une seule date de livraison par camion, au plus tard le jour du premier besoin qu'il transporte, et au plus 7 jours avant le dernier."), (34, '1.5', 'test_sortie1_structure', 'Sortie 1 : colonnes et une ligne par couple', "Le fichier existe, a les colonnes attendues dans l'ordre, et contient exactement une ligne (un profil) par couple fournisseur × usine du planning."), (35, '1.5', 'test_sortie1_valeurs', 'Sortie 1 : valeurs valides', "Identifiants au bon format, distance positive (vide seulement pour un fournisseur sans distance), fréquence de 1 à 3, jours fixes valides, distincts, dans l'ordre de la semaine et en nombre égal à la fréquence, camions par semaine positifs, remplissage entre 0 et 100 %."), (36, '1.5', 'test_sortie2_structure', 'Sortie 2 : colonnes, types et valeurs', 'Colonnes attendues, numéro de camion et nombre de lignes entiers, dates au format AAAA-MM-JJ et jour de la semaine cohérent, identifiants au bon format, quantité positive, poids renseigné sauf pour un camion réservé, et une seule ligne par material × quai dans un camion.'), (37, '1.5', 'test_sortie2_besoins_regroupes', 'Sortie 2 : besoins regroupés dans une seule cellule', 'La colonne des besoins regroupés contient, dans une seule cellule, la liste des dates de besoin du material livrées par ce camion.'), (38, '1.5', 'test_livrables_a_jour', 'Les fichiers sont à jour avec le moteur', "Les CSV relus doivent être identiques aux sorties calculées par main.py : sinon le livrable est périmé (moteur modifié sans relancer l'export).")]


class Collecteur:
    # Plugin pytest : garde le résultat de chaque cas de test
    def __init__(self):
        self.cas = []

    def pytest_runtest_logreport(self, report):
        if report.when == "call" or (report.when == "setup" and report.outcome != "passed"):
            self.cas.append((report.nodeid.split("::")[-1].split("[")[0], report.outcome == "passed"))


collecteur = Collecteur()
sortie_pytest = io.StringIO()
with contextlib.redirect_stdout(sortie_pytest):   # le détail de pytest n'est affiché qu'en cas d'échec
    code_retour = pytest.main(["-q", "--no-header", "-p", "no:cacheprovider", "../tests"], plugins=[collecteur])

# ---------- Tableau récapitulatif : un test par ligne, OK si tous ses cas passent ----------
cas = pd.DataFrame(collecteur.cas, columns=["fonction", "ok"]).groupby("fonction")["ok"].agg(["size", "all"])
bilan = pd.DataFrame(DESCRIPTIONS, columns=["N°", "Section", "fonction", "Test", "Ce qu'il vérifie"])
bilan["Cas"] = bilan["fonction"].map(cas["size"]).fillna(0).astype(int)
bilan["Résultat"] = ["OK" if f in cas.index and cas.loc[f, "all"] else "NOK" for f in bilan["fonction"]]
bilan = bilan.drop(columns="fonction")

couleurs = lambda v: ("color: #1b7f3b; font-weight: bold" if v == "OK" else "color: #b00020; font-weight: bold")
display(bilan.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "normal"})
        .map(couleurs, subset=["Résultat"]))
print(f"{(bilan['Résultat'] == 'OK').sum()} tests OK sur {len(bilan)} ({bilan['Cas'].sum()} cas). "
      + ("Tous les tests passent." if code_retour == 0 else "Des tests échouent : voir les lignes NOK."))
if code_retour != 0:
    print(sortie_pytest.getvalue())

Résultat : les 38 tests passent, soit 106 cas en comptant chaque cas des tests paramétrés.

- **Fonctions** : les fonctions clés du moteur donnent les résultats attendus sur des cas construits à la main, y compris aux bornes. La tranche exacte est retrouvée, 7 jours d'avance sont acceptés et 8 refusés, et un deuxième camion est ajouté dès que la capacité en poids ou en volume est dépassée. Le prix calculé retrouve l'exemple de la rate sheet (335,34 €), et il ne baisse jamais quand le poids ou la distance augmente. La règle du jour de départ est vérifiée pour les 25 profils et chaque jour du planning, et `profil_utilise` (C11) distingue un profil utilisé d'un profil dont un jour n'est jamais livré.
- **Contraintes** : le planning final respecte les 10 contraintes dures (C1 à C9 et C11), et le lissage reste à 2 semaines hors tolérance sur 20. Pour C11, chaque couple à 2 ou 3 jours fixes livre au moins une semaine sur tous ses jours. Chaque fonction de vérification détecte l'erreur qu'on lui introduit, donc les contrôles du milestone 3 sont fiables.
- **Données manquantes** : les hypothèses sont appliquées comme prévu. La ligne de plus de 24,5 t est exclue, chaque ligne sans poids a son camion réservé, et les fournisseurs sans distance sont chiffrés au tarif le plus élevé. Les lignes de quantité nulle ne sont pas livrées.
- **Exhaustivité et anticipation** : rien n'est perdu ni inventé. Chaque ligne est livrée une fois, et la quantité livrée est égale à la quantité demandée pour chaque material, quai et couple. Aucune usine n'est livrée en retard, ni plus de 7 jours avant son besoin.
- **Livrables** : `sortie1_profils.csv` et `sortie2_camions.csv` ont les colonnes attendues et des valeurs valides. Les besoins regroupés tiennent dans une seule cellule, et les fichiers sont à jour avec le moteur. Un fichier modifié à la main, par exemple avec une fréquence de 4, fait échouer les tests.

## 2. Analyse comparative (avant / après)

### Nouvelles distributions (taux de remplissage, coûts, ...)

### Étude des économies réalisées (réalisé vs potentiel)

## Conclusion